This notebook should reproduce paper within MFT 

M. di Volo, A. Romagnoni, C. Capone, and A. Destexhe, “Biologically Realistic Mean-Field Models of Conductance-Based Networks of Spiking Neurons with Adaptation,” Neural Computation, vol. 31, no. 4, pp. 653–680, Apr. 2019, doi: 10.1162/neco_a_01173.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import json
import time
import os
import pickle
import multiprocessing as mp
from importlib import reload


import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image

# Add the parent directory to the system path to import modules
# sys.path.append(str(Path(__file__).absolute().parent))
if 'haman' in os.getcwd():
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester')
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester/codes')
    os.chdir('/home/haman/mf-temporary/MeanFieldTester')
    project_path = Path('/home/haman/mf-temporary')
elif 'pavel' in os.getcwd():
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester')
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester/codes')
    os.chdir('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester')
    project_path = Path('/home/pavel/academia/wintermute/mf-temporary')

import codes.controller as rw
from codes.plotting import fig_plots
from codes import plotting as ax_plt
import codes.cell_library as cells
import codes.neuron_simulation as ns
import codes.meanfield_simulation as mfs
import codes.network_simulation as nets
import codes.transfer_function as tf
from codes import utils

from codes.tvb_models.models import Zerlaut_adaptation_first_order


Defining the workflow parameters. Important params to replicate the paper:
- `transfer_functions.square_terms == True`
- `transfer_functions.log_term == False`
- `transfer_functions.adaptation == True`
- `transfer_functions.fit_with_w == True`

In [ ]:
workflow_params = {
    "single_neuron_simulations" : {
        "simulate_single_neuron" : False,
        "load_single_neuron" : True,
        "fix_nu_out" : True,
        "nu_e_range" : [0, 30, 31],
        "nu_i_range" : [0, 30, 31],
        "nu_out_range" : [0, 30, 31],
        "max_nu_e_step" : 1,
        "simulation_time" : 5000,
        "averaging_window" : 2000,
        "time_step" : 0.1,
        "seed" : 42,
        "n_runs" : 5,
        "cpus" : 16
    },
    "transfer_functions" : {
        "fit_transfer_function" : False,
        "square_terms" : True,
        "log_term" : False,
        "adaptation" : True,
        "fit_with_w" : True,
        "nu_out_min" : 0.0,
        "nu_out_max" : 200.0,
        "V_eff_fitting" : {
            "method" : "SLSQP",
            "options" : {
                "ftol" : 5e-9,
                "disp" : False,
                "maxiter" : 10000
            }
        },
        "TF_fitting" : {
            "method" : "nelder-mead",
            "options" : {
                "fatol": 5e-9,
                "disp" : False,
                "maxiter" : 10000
            }
        }
    },
    "network_simulations" : {
        "load" : True,
        "timestep" : 0.1,
        "seed" : 42,
        "n_runs" : 5,
        "smoothing_function" : "sliding_window",
        "smoothing_constant" : 50,
        "smoothing_kwargs" : {}
    },
    "mf_model" : {
        "T" : 20.0,
        "tvb_model" : "DiVolo_Tsodyks_second_order",
        "mf_init" : {
            "E" : [0.000, 0.000],
            "I" : [0.00, 0.00],
            "C_ee" : [0.0000005, 0.0000005],
            "C_ei" : [0.0000005, 0.0000005],
            "C_ii" : [0.0000005, 0.0000005],
            "W_e" : [50.0, 50.0],
            "W_i" : [0.0, 0.0],
            "X_e" : [1.0, 1.0],
            "Y_e" : [0.0, 0.0],
            "X_i" : [1.0, 1.0],
            "Y_i" : [0.0, 0.0],
            "noise" : [0.0, 0.0],
            "stimulus" : [0.0, 0.0]
        }
    }
}


Defining the network params. Using parameters from the paper (neurons, network, and TF fit coefficients)

In [ ]:
network_params = {
    "exc_neuron" : {                            # taken from diVolo2019_BiologicallyRealistic
        "neuron_params" : {
            "v_rest" : -65.0,                   # [mV], Resting membrane potential
            "v_reset" : -65.0,                  # [mV], Reset potential after spike
            "tau_refrac" : 5.0,                 # [ms], Refractory period
            "tau_m" : 20.0,                     # [ms], Membrane time constant, taken as cm/gl
            "cm" : 0.200,                       # [nF], Membrane capacitance
            "e_rev_E" : 0.0,                    # [mV], Excitatory reversal potential
            "e_rev_I" : -80.0,                  # [mV], Inhibitory reversal potential
            "tau_syn_E" : 5.0,                  # [ms], Excitatory synaptic time constant
            "tau_syn_I" : 5.0,                  # [ms], Inhibitory synaptic time constant
            "a" : 4.0,                          # [nS], Subthreshold adaptation conductance
            "b" : 0.02,                         # [nA], Spike-triggered adaptation increment
            "delta_T" : 2.0,                    # [mV], Slope factor
            "tau_w" : 500.0,                    # [ms], Adaptation time constant
            "v_thresh" : -50.0,                 # [mV], Spike threshold
        },
        "init_values" : {
            "w" : 0.00,                         # [nA], Default value of adaptation current
            "v" : -65.0,                        # [mV], Default value of membrane potential
        },
    },
    "inh_neuron" : {                            # taken from diVolo2019_BiologicallyRealistic
        "neuron_params" : {
            "v_rest" : -65.0,                   # [mV], Resting membrane potential
            "v_reset" : -65.0,                  # [mV], Reset potential after spike
            "tau_refrac" : 5.0,                 # [ms], Refractory period
            "tau_m" : 20.0,                     # [ms], Membrane time constant, taken as cm/gl
            "cm" : 0.200,                       # [nF], Membrane capacitance
            "e_rev_E" : 0.0,                    # [mV], Excitatory reversal potential
            "e_rev_I" : -80.0,                  # [mV], Inhibitory reversal potential
            "tau_syn_E" : 5.0,                  # [ms], Excitatory synaptic time constant
            "tau_syn_I" : 5.0,                  # [ms], Inhibitory synaptic time constant
            "a" : 0.0,                          # [nS], Subthreshold adaptation conductance
            "b" : 0.0,                          # [nA], Spike-triggered adaptation increment
            "delta_T" : 0.5,                    # [mV], Slope factor
            "tau_w" : 500.0,                    # [ms], Adaptation time constant
            "v_thresh" : -50.0,                 # [mV], Spike threshold
        },
        "init_values" : {
            "w" : 0.0,                          # [nA], Default value of adaptation current
            "v" : -65.0,                        # [mV], Default value of membrane potential
        },
    },
    "network" : {
        "total_pop_size" : 10000,                  # int,Total number of exc and inh neurons
        "drive_pop_size" : 8000,                     # int, number of driving neurons
        "stim_pop_size" : 8000,
        "g" : 0.2,                                  # float in [0,1], percentage of inhibitory neurons
        "p_connect_exc" : 0.05,                       # Probability of incoming connections coming from excitatory neurons
        "p_connect_inh" : 0.05,                       # Probability of incoming connections coming from inhibitory neurons
        "p_connect_drive" : 0.05,
        "p_connect_stim" : 0.05,
    },
    "synapses" : {
        "exc_synapses" : {
            "syn_params" : {
                "weight" : 1.0,                     # [nS], synaptic weight
                "delay" : 0.1,                      # [ms], synaptic delay
            },
            "syn_type" : "static_synapse",          # str, nest synapse type
        },
        "inh_synapses" : {
            "syn_params" : {
                "weight" : 5.0,                     # [nS], synaptic weight
                "delay" : 0.1,                      # [ms], synaptic delay
            },
            "syn_type" : "static_synapse",          # str, nest synapse type
        },
        "drive_synapses" : {
            "syn_params" : {
                "weight" : 1.0,                     # [nS], synaptic weight
                "delay" : 0.1,                      # [ms], synaptic delay
            },
            "syn_type" : "static_synapse",         # str, nest synapse type
        },
        "stim_synapses" : {
            "syn_params" : {
                "weight" : 1.0,                     # [nS], synaptic weight
                "delay" : 0.1,                      # [ms], synaptic delay
            },
            "syn_type" : "static_synapse",        # str, nest synapse type
        },
    },
    "transfer_function" : {
        "expansion_point" : [-60, 4, 0.5],
        "expansion_norm" : [10, 6, 1.0],

        # Di Volo 2019 paper
        # "exc_neuron" : [-49.8, 5.06, -25.0, 1.4, -0.41, 10.5, -36.0, 7.4, 1.2, -40.7],      # [mV], coefficients of the polynomial
        # "inh_neuron" : [-51.4, 4.0, -8.3, 0.2, -0.5, 1.4, -14.6, 4.5, 2.8, -15.3],          # [mV], coefficients of the polynomial

        # Di Volo repo
        "exc_neuron" : [-49.83106, 5.06355, -23.47012, 2.29515, -0.41053, 10.54705, -36.59253, 7.43749, 1.26506, -40.72161],      # [mV], coefficients of the polynomial
        "inh_neuron" : [-51.49122, 4.00369, -8.35201, 0.24142, -0.50706, 1.43454, -14.68669, 4.50271, 2.84722, -15.3578],          # [mV], coefficients of the polynomial
    }
}

In [ ]:
stimuli_params = {
    "SpontActivity" : {
        "pattern" : "NoStimulus",
        "stim_pars" : {},
        "drive_rate" : 2.5,
        "drive_increase_duration" : 400,
        "stim_target_ratio" : 1.0,
        "simulation_time" : 2000,
        "target_nodes" : 0,
        "direct_stimulation" : False
    },
}

In [ ]:
runner = rw.WorkflowRunner(sim_name="DiVolo2019",
                           snn_params=network_params,
                           workflow_params=workflow_params,
                           stimuli_params=stimuli_params,
                           neuron_data_file_mark="divolo-static",
                           results_dir=project_path / "projects" / "02_replicating_divolo" / "results",
                           )

runner.add_default_mf_model(mf_name="Paper-Fit")

runner.add_mf_model(
    mf_name="My-Fit",
    network_params = network_params,
    mf_model_pars={
        "tvb_model" : "Zerlaut_adaptation_second_order",
    },
    tf_sim_pars={
        "adaptation" : True,
        "fit_transfer_function" : True,
    },
)



In [ ]:
runner.simulate_neurons(plot=True, save=True)
Image(filename=str(runner.results_path) +'/neuron_activity.png')

In [ ]:
runner.fit_transfer_functions(plot=True)
Image(filename=str(runner.results_path) +'/tf_fits.png') 

In [ ]:
fig_plots.fig_tf_fits_together(
    runner.neuron_results, 
    runner.tf_funcs, 
    common_params={
        'labels' : runner.mf_names,
        'linestyles' : ['--', ':'],
        'xlim' : (0,7),
        'ylim' : (0, 35),
    }, 
    fig_params={
        'fontsize': 14,
        'figsize': (15, 10),  # width, height
        'tight_layout': True,
        'savefig': True,
        'savefig_path': runner.results_path / f"tf_fits2.png",
    })

Image(filename=str(runner.results_path) +'/tf_fits2.png') 

# Fig 1
To ensure I am not having errors in my MFT code, I also compare with di Volo code. Here I use the scripts from ModelDB repo for diVolo paper

https://github.com/ModelDBRepository/263236/tree/master

In [ ]:
import numpy as np
import numba
import scipy.special as sp_spec
import scipy.integrate as sp_int
from scipy.optimize import minimize, curve_fit
import sys
import matplotlib.pyplot as plt
from scipy import signal
from scipy.integrate import quad


def pseq_params(params):
    Qe, Te, Ee = params['Qe'], params['Te'], params['Ee']
    Qi, Ti, Ei = params['Qi'], params['Ti'], params['Ei']
    Gl, Cm , El = params['Gl'], params['Cm'] , params['El']
    for key, dval in zip(['Ntot', 'pconnec', 'gei'], [1, 2., 0.5]):
        if key in params.keys():
            exec(key+' = params[key]')
        else: # default value
            exec(key+' = dval')
    if 'P' in params.keys():
        P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10 = params['P']
    else: # no correction
        P0 = -45e-3
        for i in range(1,11):
            exec('P'+str(i)+'= 0')

    return Qe,Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10

def get_fluct_regime_vars(Fe, Fi, XX,Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10):
    # here TOTAL (sum over synapses) excitatory and inhibitory input
    fe = Fe*(1.-gei)*pconnec*Ntot # default is 1 !!
    
    fi = Fi*gei*pconnec*Ntot
    muGe, muGi = Qe*Te*fe, Qi*Ti*fi
    muG = Gl+muGe+muGi
    muV = (muGe*Ee+muGi*Ei+Gl*El-XX)/muG
    muGn, Tm = muG/Gl, Cm/muG
    
    Ue, Ui = Qe/muG*(Ee-muV), Qi/muG*(Ei-muV)

    sV = np.sqrt(\
                 fe*(Ue*Te)**2/2./(Te+Tm)+\
                 fi*(Ti*Ui)**2/2./(Ti+Tm))

    fe, fi = fe+1e-9, fi+1e-9 # just to insure a non zero division,
   
    Tv = ( fe*(Ue*Te)**2 + fi*(Ti*Ui)**2 ) /( fe*(Ue*Te)**2/(Te+Tm) + fi*(Ti*Ui)**2/(Ti+Tm) )
    TvN = Tv*Gl/Cm

    return muV, sV+1e-12, muGn, TvN

def mean_and_var_conductance(Fe, Fi, Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10):
    # here TOTAL (sum over synapses) excitatory and inhibitory input
    fe = Fe*(1.-gei)*pconnec*Ntot # default is 1 !!
    fi = Fi*gei*pconnec*Ntot
    return Qe*Te*fe, Qi*Ti*fi, Qe*np.sqrt(Te*fe/2.), Qi*np.sqrt(Ti*fi/2.)


### FUNCTION, INVERSE FUNCTION
def erfc_func(muV, sV, TvN, Vthre, Gl, Cm):
    return .5/TvN*Gl/Cm*(sp_spec.erfc((Vthre-muV)/np.sqrt(2)/sV))

def effective_Vthre(Y, muV, sV, TvN, Gl, Cm):
    Vthre_eff = muV+np.sqrt(2)*sV*sp_spec.erfcinv(\
                    Y*2.*TvN*Cm/Gl) # effective threshold
    return Vthre_eff

def threshold_func(muV, sV, TvN, muGn, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10):
    """
    setting by default to True the square
    because when use by external modules, coeff[5:]=np.zeros(3)
    in the case of a linear threshold
    """
    
    muV0, DmuV0 = -60e-3,10e-3
    sV0, DsV0 =4e-3, 6e-3
    TvN0, DTvN0 = 0.5, 1.
    
    return P0+P1*(muV-muV0)/DmuV0+\
        P2*(sV-sV0)/DsV0+P3*(TvN-TvN0)/DTvN0+\
        0*P4*np.log(muGn)+P5*((muV-muV0)/DmuV0)**2+\
        P6*((sV-sV0)/DsV0)**2+P7*((TvN-TvN0)/DTvN0)**2+\
        P8*(muV-muV0)/DmuV0*(sV-sV0)/DsV0+\
        P9*(muV-muV0)/DmuV0*(TvN-TvN0)/DTvN0+\
        P10*(sV-sV0)/DsV0*(TvN-TvN0)/DTvN0
      
# final transfer function template :

def TF_my_template(fe, fi,XX, Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10):
    # here TOTAL (sum over synapses) excitatory and inhibitory input
    if(hasattr(fe, "__len__")):
        fe[fe<1e-8]=1e-8
    else:
        if(fe<1e-8):
            fe=1e-8
    if(hasattr(fi, "__len__")):
        fi[fi<1e-8]=1e-8
    else:
        if(fi<1e-8):
            fi=1e-8
    muV, sV, muGn, TvN = get_fluct_regime_vars(fe, fi,XX, Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10)
    Vthre = threshold_func(muV, sV, TvN, muGn, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10)

    if(hasattr(muV, "__len__")):
        #print("ttt",isinstance(muV, list), hasattr(muV, "__len__"))
        sV[sV<1e-4]=1e-4
    else:
        if(sV<1e-4):
            sV=1e-4
    
    Fout_th = erfc_func(muV, sV, TvN, Vthre, Gl, Cm)
    if(hasattr(Fout_th, "__len__")):
        #print("ttt",isinstance(muV, list), hasattr(muV, "__len__"))
        Fout_th[Fout_th<1e-8]=1e-8
    
    else:
        if(Fout_th<1e-8):
            Fout_th=1e-8
    '''
    if(El<-0.063):
    
        if(hasattr(Fout_th, "__len__")):
            #print("ttt",isinstance(muV, list), hasattr(muV, "__len__"))
            Fout_th[Fout_th>80.]=175
    
    
        else:
        
            if(Fout_th>80.):
                print("Done")
                Fout_th=175

    '''
    #print 'yyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyy',fe,fi,muV, sV, TvN,Fout_th
    return Fout_th



def gaussian(x, mu, sig):
    return (1/(sig*np.sqrt(2*3.1415)))*np.exp(-np.power(x - mu, 2.) / (2 * np.power(sig, 2.)))
def TF_my_templateup_heterogeneity(fe, fi,XX, Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10):
    # here TOTAL (sum over synapses) excitatory and inhibitory input
    def Phet(k):
        locale=gaussian(k,1.,0.2)*TF_my_template(fe, fi,XX, Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El*k, Ntot, pconnec, gei, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10)
        return locale
    outhet, err = quad(Phet, 0.1, 5)
    return outhet
    
def make_loop(t, nu, vm, nu_aff_exc, nu_aff_inh, BIN,\
              Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10):
    dt = t[1]-t[0]
    # constructing the Euler method for the activity rate
    for i_t in range(len(t)-1): # loop over time
        
        fe = (nu_aff_exc[i_t]+nu[i_t]+Fdrive) # afferent+recurrent excitation
        fi = nu[i_t]+nu_aff_inh[i_t] # recurrent inhibition
        W[i_t+1] = W[i_t] + dt/Tw*(b*nu[i_t]*Tw - W[i_t])

        nu[i_t+1] = nu[i_t] +\
               dt/BIN*(\
                TF_my_template(fe, fi, W[i_t], Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10)\
                -nu[i_t])

        vm[i_t], _, _, _ = get_fluct_regime_vars(fe, fi, W[i_t], Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10)

    return nu, vm, W


################################################################
##### Now fitting to Transfer Function data
################################################################


def fitting_Vthre_then_Fout(Fout, Fe_eff, fiSim,w, params,\
                               maxiter=50000, xtol=1e-5, with_square_terms=False):


    Gl, Cm , El = params['Gl'], params['Cm'] , params['El']

    Fout, Fe_eff, fiSim,w = Fout.flatten(), Fe_eff.flatten(), fiSim.flatten(), w.flatten()
    #print 'Eccolo', Fout[8],w[8],len(Fout)
    
    muV, sV, muGn, TvN = get_fluct_regime_vars(Fe_eff, fiSim,w, *pseq_params(params))
    i_non_zeros = np.where((Fout>0.)&(Fout<60.))

    Vthre_eff = effective_Vthre(Fout[i_non_zeros], muV[i_non_zeros],\
                sV[i_non_zeros], TvN[i_non_zeros], params['Gl'], params['Cm'])
    
    if with_square_terms:
        P = np.zeros(11)
    else:
        P = np.zeros(5)
    P[:5] = Vthre_eff.mean(), 1e-3, 1e-3, 1e-3, 1e-3

    def Res(p):
        if not with_square_terms:
            pp = np.concatenate([p, np.zeros(6)])
        else:
            pp=p
        vthre = threshold_func(muV[i_non_zeros], sV[i_non_zeros],\
                               TvN[i_non_zeros], muGn[i_non_zeros], *pp)
        return np.mean((Vthre_eff-vthre)**2)
            
    #xtol=1e-19
    #plsq = minimize(Res, P, method='nelder-mead',options={'xtol': xtol, 'disp': True, 'maxiter':maxiter})
    plsq = minimize(Res, P, method='SLSQP',options={'ftol': 1e-15, 'disp': True, 'maxiter':40000})
    #print plsq

    P = plsq.x
    
    def Res(p):
        if not with_square_terms:
            params['P'] = np.concatenate([p, np.zeros(6)])
        else:
            params['P'] = p
        return np.mean((Fout-\
                        TF_my_template(Fe_eff, fiSim,w, *pseq_params(params)))**2)

    plsq = minimize(Res, P, method='nelder-mead',\
            options={'xtol': xtol, 'disp': True, 'maxiter':maxiter})



    params['P'] = P
    
    diff=(TF_my_template(Fe_eff, fiSim,w, *pseq_params(params))-Fout).mean()
    diff_M=(TF_my_template(Fe_eff, fiSim,w, *pseq_params(params))-Fout).max()
    print("rrrrr",diff,diff_M)
          
          
  
    plt.plot(fiSim,Fout,'rd',fiSim,TF_my_template(Fe_eff, fiSim,w, *pseq_params(params)),'bs')
    #plt.plot(fiSim,Fe-eff,Fout,'rd')
    plt.show()
    thrplot=threshold_func(muV, sV,TvN, muGn, *(plsq.x))
    
    #np.save('FScell_Voltage.npy',[muV, sV,TvN,Fout])
   
    plt.plot(muV,Fout,'rd',muV,erfc_func(muV, sV, TvN, thrplot, Gl, Cm),'bd')
    plt.show()
    plt.plot(muV,erfc_func(muV, 4e-3, 0.5, thrplot, Gl, Cm),'bd')
    plt.show()



    if with_square_terms:
        #return plsq.x
        return P
    else:
        return np.concatenate([plsq.x, np.zeros(6)])

def make_fit_from_data(DATA, with_square_terms=False):

    MEANfreq, SDfreq, Fe_eff, fiSim, params,w = np.load(DATA)
    

    Fe_eff, Fout = np.array(Fe_eff), np.array(MEANfreq)
    levels = fiSim # to store for colors
    fiSim = np.meshgrid(np.zeros(Fe_eff.shape[1]), fiSim)[1]

    P = fitting_Vthre_then_Fout(Fout, Fe_eff, fiSim,w, params,\
                                with_square_terms=with_square_terms)
                                
    print("ffffff",P)

    #plt.plot(Fe_eff[2,:],MEANfreq[2,:],"bs",fiSim[:,5],MEANfreq[:,5],"o")
    plt.plot(Fe_eff[2,:],MEANfreq[2,:],"bs",fiSim[:,5],MEANfreq[:,5],"o")
    #np.save('FScell_freq.npy',[Fe_eff[2,:],fiSim[:,5],MEANfreq[:,5]])

    plt.show()
                            
    print( '==================================================')
    print (1e3*np.array(P), 'mV')

    # then we save it:
    filename = DATA.replace('.npy', '_fit.npy')
    print('coefficients saved in ', filename)
    np.save(filename, np.array(P))

    return P

In [ ]:
def find_closest_index(arr, target):
    # Ensure the input is a NumPy array
    arr = np.asarray(arr)
    
    # Calculate the absolute difference between the target and all elements
    differences = np.abs(arr - target)
    
    # Find the index of the minimum difference
    closest_idx = differences.argmin()
    closest_num = arr[closest_idx]
    
    # Check if we have an exact match
    if closest_num == target:
        return closest_idx
    else:
        # Print the requested warnings and information
        print("WARNING: Exact match not found in the array.")
        print(f"Number wanted: {target}")
        print(f"Closest number found: {closest_num}")
        return closest_idx

In [ ]:
fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(16, 8))

inh_input = 8  # Hz

######

for i, (neuron, color) in enumerate(zip(['exc_neuron', 'inh_neuron'], ['green', 'red'])):
    # 1. data points
    inh_idx = find_closest_index(runner.neuron_results['exc_neuron'].nu_i[0], inh_input)
    nu_inh = runner.neuron_results['exc_neuron'].nu_i[:,inh_idx]

    nu_out_mean = runner.neuron_results[neuron].nu_out_mean[:,inh_idx]
    nu_out_std = runner.neuron_results[neuron].nu_out_std[:,inh_idx]
    nu_exc = runner.neuron_results[neuron].nu_e[:,inh_idx]
    w_mean = runner.neuron_results[neuron].w_mean[:,inh_idx]
    voltage_mean = runner.neuron_results[neuron].voltage_mean[:,inh_idx]
    voltage_std = runner.neuron_results[neuron].voltage_std[:,inh_idx]

    ax1.errorbar(nu_exc, nu_out_mean, yerr=nu_out_std, color=color, fmt="o", label=neuron.replace('_', ' ')+" data")
    

    # 2. MFT TF curves 
    for mf_name, tf_func, ls in zip(runner.mf_names, runner.tf_funcs[neuron], ["--", ":"]):
        if i== 0:
            ax1.plot(nu_exc, tf_func(nu_exc, inh_input, w_mean*1e-3, flattened=True), ls, color=color, label=mf_name)
        else:
            ax1.plot(nu_exc, tf_func(nu_exc, inh_input, w_mean*1e-3, flattened=True), ls, color=color)

    # 3. di Volo code TF curves
    Qe = runner.mf_net_pars[0]['synapses']['exc_synapses']['syn_params']['weight'] *1e-9
    Te = runner.mf_net_pars[0][neuron]['neuron_params']['tau_syn_E'] * 1e-3
    Ee = runner.mf_net_pars[0][neuron]['neuron_params']['e_rev_E'] * 1e-3

    Qi = runner.mf_net_pars[0]['synapses']['inh_synapses']['syn_params']['weight'] * 1e-9
    Ti = runner.mf_net_pars[0][neuron]['neuron_params']['tau_syn_I'] * 1e-3
    Ei = runner.mf_net_pars[0][neuron]['neuron_params']['e_rev_I'] * 1e-3

    Cm = runner.mf_net_pars[0][neuron]['neuron_params']['cm'] * 1e-9
    Gl = Cm / (runner.mf_net_pars[0][neuron]['neuron_params']['tau_m'] * 1e-3)
    El = runner.mf_net_pars[0][neuron]['neuron_params']['v_rest'] * 1e-3

    Ntot = runner.mf_net_pars[0]['network']['total_pop_size']
    pconnec = runner.mf_net_pars[0]['network']['p_connect_exc']
    gei = runner.mf_net_pars[0]['network']['g']

    P0, P1, P2, P3, P5, P6, P7, P8, P9, P10 = np.array(runner.tf_funcs[neuron][0].v_eff.coefs)*1e-3
    P4 = 0.0 # no log term
    if i == 0:
        ax1.plot(nu_exc, TF_my_template(
                                    nu_exc, 
                                    inh_input,
                                    w_mean*1e-12,
                                    Qe, 
                                    Te, 
                                    Ee, 
                                    Qi, 
                                    Ti, 
                                    Ei, 
                                    Gl, 
                                    Cm, 
                                    El, 
                                    Ntot, 
                                    pconnec, 
                                    gei,
                                    P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10
                                    ),
                    "-", color=color, alpha=0.3, linewidth=6, label="di Volo code"
        )
    else:
        ax1.plot(nu_exc, TF_my_template(
                                    nu_exc, 
                                    inh_input,
                                    w_mean*1e-12,
                                    Qe, 
                                    Te, 
                                    Ee, 
                                    Qi, 
                                    Ti, 
                                    Ei, 
                                    Gl, 
                                    Cm, 
                                    El, 
                                    Ntot, 
                                    pconnec, 
                                    gei,
                                    P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10
                                    ),
                    "-", color=color, alpha=0.3, linewidth=6
        )
    # DiVolo code
    # fit params P0 - P10 are provided, but not used at all
    muV1, *_ = get_fluct_regime_vars(nu_exc, inh_input, w_mean*1e-12,Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10)
    if i == 0:
        ax2.plot(nu_exc, muV1*1e3, "-", color=color, alpha=0.3, linewidth=6, label="di Volo code")
    else:
        ax2.plot(nu_exc, muV1*1e3, "-", color=color, alpha=0.3, linewidth=6)

    # my code
    # MeanPotentialFluctuations
    # MPF_with_nu_out
    muV2, *_ = tf.MPF_with_adaptation(runner.mf_net_pars[0][neuron])(nu_exc, inh_input, w_mean*1e-3, flattened=True)
    if i == 0:
        ax2.plot(nu_exc, muV2, "--", color=color, label="MPF with adaptation")
    else:
        ax2.plot(nu_exc, muV2, "--", color=color)

    # ax2.errorbar(nu_exc, voltage_mean, yerr=voltage_std, color=color, fmt="o", label=neuron.replace('_', ' ')+" voltage")
    ax2.plot(nu_exc, voltage_mean, "o", color=color, label=neuron.replace('_', ' ')+" voltage")

    print((np.abs(muV1*1e3-muV2)).mean())



# TVB MPF?

ax1.legend()
ax1.set_xlabel("nu_exc (Hz)")
ax1.set_ylabel("nu_out (Hz)")
ax1.set_title(f"Fig 1A: Transfer Function at nu_inh={inh_input:.1f} Hz")
ax1.set_xlim(0, 7)
ax1.set_ylim(0, 35)
ax1.tick_params(right=True, labelright=True)

ax2.set_title(f"Fig 1B: Voltage at nu_inh={inh_input:.1f} Hz")
ax2.set_xlim(0, 7)
ax2.set_ylim(-75,-45)
ax2.tick_params(right=True, labelright=True)
ax2.set_xlabel("nu_exc (Hz)")
ax2.set_ylabel("Voltage (mV)")
ax2.legend()

fig.tight_layout()
plt.show()

This figuere is very different from the paper fig, the values it reaches

# Fig 2

This fig requires network simulations and inspection of a parameter (adaptation parameter b)

In [ ]:
adaptation_inspection_results = runner.inspect_spont_activity(
    # param_values=[0.02, 0.05, 0.1, 0.2],
    param_values=[0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.10, 0.11, 0.12, 0.13, 0.14, 0.15],
    param_to_inspect="network.exc_neuron.neuron_params.b",
    stimulus={'drive_rate':2.5}
)

In [ ]:
fig, axes = plt.subplots(ncols=3, figsize=(16, 8))

plot = ax_plt.FiringRateInspectionPlot({
    "markers": ["o"]+[""]* (len(adaptation_inspection_results)-1),
    "labels": [result.inspected_network_name for result in adaptation_inspection_results],
    "linestyles": [""] + [ ':', '--'],
    "legend": True,
    "xlabel": "b (nA)"
})
plot.draw(axes[0], adaptation_inspection_results)

plot = ax_plt.VoltageInspectionPlot({
    "markers": ["o"]+[""]* (len(adaptation_inspection_results)-1),
    "labels": [result.inspected_network_name for result in adaptation_inspection_results],
    "linestyles": [""] + [ ':', '--'],
    "legend": True,
    "xlabel": "b (nA)"
})
plot.draw(axes[1], adaptation_inspection_results)

plot = ax_plt.AdaptationInspectionPlot({
    "markers": ["o"]+[""]* (len(adaptation_inspection_results)-1),
    "labels": [result.inspected_network_name for result in adaptation_inspection_results],
    "linestyles": [""] + [ ':', '--'],
    "legend": True,
    "xlabel": "b (nA)"
})
plot.draw(axes[2], adaptation_inspection_results)


fig.suptitle("Spontaneous Activity Inspection Results")
fig.tight_layout()
plt.show()